# Create The Health Foundation Awards

**The Health Foundation** (funder_id `4320320265`, UK, IAMHRF, priority `353`).
health.org.uk projects A-Z (628 projects, 2007-2025) via Firefox-headless (Akamai
blocks plain requests + Chromium; no bulk export exists — checked 360Giving/CKAN).
Org-level: title + programme + year + summary; NO PI, NO amounts, NO institution
at source (§6.7) — lead_investigator NULL.

Parquet: `s3://openalex-ingest/awards/health_foundation/health_foundation_grants.parquet`

## Step 1: Create Staging Table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.health_foundation_raw
USING delta
AS
SELECT *, current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/health_foundation/health_foundation_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) as total_projects FROM openalex.awards.health_foundation_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.health_foundation_raw LIMIT 5;

## Step 2: Create Health Foundation Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.health_foundation_awards
USING delta
AS
WITH
thf_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder WHERE funder_id = 4320320265
),
awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', LOWER(g.funder_award_id)))) % 9000000000 as id,
        g.title as display_name,
        g.description as description,
        f.funder_id,
        g.funder_award_id as funder_award_id,
        CAST(NULL AS DECIMAL(18,2)) as amount,
        CAST(NULL AS STRING) as currency,
        struct(CONCAT('https://openalex.org/F', f.funder_id) as id, f.display_name, f.ror_id, f.doi) as funder,
        'grant' as funding_type,
        g.scheme as funder_scheme,
        'health_foundation' as provenance,
        CAST(NULL AS DATE) as start_date,
        CAST(NULL AS DATE) as end_date,
        TRY_CAST(g.year_awarded AS INT) as start_year,
        CAST(NULL AS INT) as end_year,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as lead_investigator,
        CAST(NULL AS STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>) as co_lead_investigator,
        CAST(NULL AS ARRAY<STRUCT<given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE, affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>>>) as investigators,
        g.landing_page_url,
        CAST(NULL AS STRING) as doi,
        CAST(NULL AS STRING) as works_api_url,
        current_timestamp() as created_date,
        current_timestamp() as updated_date
    FROM openalex.awards.health_foundation_raw g
    CROSS JOIN thf_funder f
)
SELECT * FROM awards_transformed;

## Step 3: Insert into openalex_awards_raw

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw WHERE provenance = 'health_foundation' AND priority = 353;
INSERT INTO openalex.awards.openalex_awards_raw
SELECT id, display_name, description, funder_id, funder_award_id, amount, currency,
    funder, funding_type, funder_scheme, provenance, start_date, end_date,
    start_year, end_year, lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url, created_date, updated_date, 353 as priority
FROM openalex.awards.health_foundation_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) total, SUM(CASE WHEN display_name IS NULL OR LENGTH(TRIM(display_name))=0 THEN 1 ELSE 0 END) blank,
  COUNT(DISTINCT funder_award_id) uniq, COUNT(start_year) has_year, MIN(start_year) miny, MAX(start_year) maxy
FROM openalex.awards.health_foundation_awards;

In [ ]:
%sql
SELECT COUNT(*) as in_raw FROM openalex.awards.openalex_awards_raw WHERE provenance='health_foundation' AND priority=353;